# Clase 034 — Parquet, Arrow, PyArrow, DuckDB

Stack columnar moderno: generamos un dataset sintético, lo escribimos en CSV vs Parquet (varias compresiones), inspeccionamos metadata, y consultamos con DuckDB sin cargar a RAM.

In [ ]:
import time, tempfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    import duckdb
except ImportError as e:
    raise ImportError('Instalá: pip install pyarrow duckdb pandas') from e

print(f'pyarrow {pa.__version__} | duckdb {duckdb.__version__} | pandas {pd.__version__}')

## 1. Dataset sintético (~1M filas, varias columnas tipadas)

In [ ]:
rng = np.random.default_rng(42)
N = 1_000_000

df = pd.DataFrame({
    'id': np.arange(N, dtype=np.int64),
    'fecha': pd.date_range('2024-01-01', periods=N, freq='min'),
    'tienda': rng.integers(1, 50, N).astype(np.int32),
    'producto': rng.choice([f'P{i:04d}' for i in range(200)], N),
    'cantidad': rng.integers(1, 20, N).astype(np.int32),
    'precio': rng.gamma(2.0, 50.0, N).astype(np.float32).round(2),
    'descuento': rng.uniform(0, 0.3, N).astype(np.float32).round(3),
})
print(df.dtypes)
print(f'shape: {df.shape}')

## 2. CSV vs Parquet (snappy / zstd / gzip): tamaño en disco

In [ ]:
tmpdir = Path(tempfile.mkdtemp(prefix='parquet_demo_'))

paths = {}
paths['csv'] = tmpdir / 'data.csv'
df.to_csv(paths['csv'], index=False)

for comp in ['snappy', 'zstd', 'gzip']:
    p = tmpdir / f'data_{comp}.parquet'
    df.to_parquet(p, compression=comp, engine='pyarrow')
    paths[f'parquet-{comp}'] = p

for k, p in paths.items():
    size_mb = p.stat().st_size / 1e6
    print(f'{k:20s} → {size_mb:7.2f} MB')

## 3. Benchmark de lectura

In [ ]:
def bench(label, fn, n=3):
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    print(f'{label:30s} → {min(ts):.3f}s (best of {n})')

bench('pandas read_csv',      lambda: pd.read_csv(paths['csv']))
bench('pandas read_parquet snappy', lambda: pd.read_parquet(paths['parquet-snappy']))
bench('pandas read_parquet zstd',   lambda: pd.read_parquet(paths['parquet-zstd']))
bench('pyarrow read_table  zstd',   lambda: pq.read_table(paths['parquet-zstd']))

## 4. Column pruning (leer solo 2 cols de 7)

In [ ]:
t0 = time.perf_counter()
full = pq.read_table(paths['parquet-zstd'])
t_full = time.perf_counter() - t0

t0 = time.perf_counter()
subset = pq.read_table(paths['parquet-zstd'], columns=['tienda', 'precio'])
t_sub = time.perf_counter() - t0

print(f'todo ({full.num_columns} cols): {t_full*1000:.1f} ms')
print(f'pruning (2 cols)        : {t_sub*1000:.1f} ms')
print(f'speedup: {t_full/t_sub:.2f}x')

## 5. Inspeccionar metadata Parquet (row groups, statistics)

In [ ]:
meta = pq.read_metadata(paths['parquet-zstd'])
print(f'num_rows      : {meta.num_rows:,}')
print(f'num_columns   : {meta.num_columns}')
print(f'num_row_groups: {meta.num_row_groups}')
print(f'format_version: {meta.format_version}')
print(f'created_by    : {meta.created_by}')

rg0 = meta.row_group(0)
print(f'\n--- row group 0 ---')
print(f'num_rows: {rg0.num_rows:,}')
for i in range(meta.num_columns):
    col = rg0.column(i)
    s = col.statistics
    rng_str = f'{s.min}..{s.max}' if s and s.has_min_max else 'n/a'
    print(f'  col[{i}] {col.path_in_schema:12s} comp={col.compression:8s} | min..max = {rng_str}')

## 6. DuckDB sobre Parquet — SQL sin cargar a RAM

In [ ]:
pq_path = str(paths['parquet-zstd']).replace('\\', '/')

t0 = time.perf_counter()
ddb = duckdb.sql(f"""
    SELECT tienda,
           COUNT(*)                    AS n_ventas,
           SUM(cantidad * precio)      AS facturado,
           AVG(precio)                 AS precio_medio
      FROM '{pq_path}'
     WHERE precio > 50
  GROUP BY tienda
  ORDER BY facturado DESC
     LIMIT 10
""").df()
t_duck = time.perf_counter() - t0

t0 = time.perf_counter()
tmp = pd.read_parquet(paths['parquet-zstd'])
tmp = tmp[tmp['precio'] > 50]
tmp = (tmp.assign(rev=tmp['cantidad'] * tmp['precio'])
          .groupby('tienda')
          .agg(n_ventas=('id', 'count'), facturado=('rev', 'sum'), precio_medio=('precio', 'mean'))
          .sort_values('facturado', ascending=False).head(10))
t_pd = time.perf_counter() - t0

print(f'duckdb (sobre parquet directo): {t_duck:.3f}s')
print(f'pandas (read + filter + agg)  : {t_pd:.3f}s')
print(f'speedup: {t_pd/t_duck:.2f}x')
print(ddb)

## 7. Schema evolution — agregar columna sin reescribir el original

In [ ]:
# Patrón: nuevo Parquet con columna extra; el lector tolera schemas distintos por archivo
df_v2 = df.head(100_000).copy()
df_v2['canal'] = rng.choice(['online', 'tienda', 'mayorista'], len(df_v2))
df_v2.to_parquet(tmpdir / 'data_v2.parquet', compression='zstd')

schema_v1 = pq.read_schema(paths['parquet-zstd'])
schema_v2 = pq.read_schema(tmpdir / 'data_v2.parquet')

print('--- schema v1 ---')
print(schema_v1)
print('\n--- schema v2 (con canal) ---')
print(schema_v2)

# DuckDB lee ambos con union_by_name aunque difieran columnas
merged = duckdb.sql(f"""
    SELECT * FROM read_parquet(['{pq_path}', '{str(tmpdir / 'data_v2.parquet').replace(chr(92), '/')}'], union_by_name=true)
    LIMIT 3
""").df()
print('\n--- merged sample (canal aparece como NaN en filas de v1) ---')
print(merged[['id', 'tienda', 'precio', 'canal']].head(3))
print('\nnulls en canal:', merged['canal'].isna().sum(), '/', len(merged))

## 8. Arrow zero-copy: pandas ↔ pyarrow ↔ duckdb

In [ ]:
# pandas → arrow → duckdb sin pasar por disco
small = df.head(50_000)
arrow_tbl = pa.Table.from_pandas(small)

result = duckdb.sql('SELECT tienda, AVG(precio) AS p_medio FROM arrow_tbl GROUP BY tienda ORDER BY p_medio DESC LIMIT 5').df()
print(result)
print(f'\narrow table: {arrow_tbl.num_rows:,} filas, {arrow_tbl.nbytes / 1e6:.2f} MB en RAM')

## 9. Cleanup

In [ ]:
shutil.rmtree(tmpdir, ignore_errors=True)
print(f'limpieza ok: {tmpdir} eliminado')

## Ejercicios

1. Particioná `df` por `tienda` con `df.to_parquet('out/', partition_cols=['tienda'])` y mirá la estructura de directorios.
2. Levantá un Parquet particionado con DuckDB usando glob `'out/**/*.parquet'`.
3. Probá `compression='brotli'` y compará tamaño vs zstd.
4. Convertí el dataset a Polars y compará tiempo de query con DuckDB.
5. Bonus: usá `pq.write_to_dataset` para escribir particionado nativo de pyarrow.